### Agent

<small>User prompt: "Schedule a meeting with Joe next week and send him a confirmation email."

To complete this, the LLM with help of Agent will do the following tasks:  
- Check your calendar
- Find free time slots
- Check Joe’s availability
- Create the calendar event
- Send a confirmation email
- Ask follow-up questions if information is missing</small>

**Agent Loop**  
<small>It repeatedly performs:  
- Reason  
- Choose an action  
- Call a tool  
- Observe result  
- Update plan  
- Repeat until done</nsmall>

### ReAct pattern(Reason + Act)

```mermaid
---
title: ReAct Pattern
---
flowchart LR
    A[User Query] --> B(REason \n Analyze & Decide)
    B --> C>Task Complete?]
    C -->|Yes| D[Final Result]
    C -->|No| E[ACTion \nUse Tools & Execute]
    E --> F[Observation \n review result]
    F --> B
```

<small>ReAct pattern combines:  
- Reasoning steps (“Thought”)  
- Actions (“Tool Calls”)  
- Observations (“Tool Results”)  

**Typical Flow**:  

Thought: I need weather data  
Action: call_weather_api("Bangalore")  
Observation: Rainy, 24°C  

Thought: Need calendar availability  
Action: call_calendar_api()  
Observation: Free at 4 PM  

Thought: Enough information gathered  
Final Answer: Schedule outdoor meeting tomorrow instead  

The model alternates between:  
Internal reasoning  
External actions</nsmall>

### Tools

<small>Tools are external functions/APIs the agent can use.  
Examples:  
Weather API  
Calendar API   
Search engine   
Email sender</nsmall>

<small>How LLM selects the appropriate tool?  
The LLM reads:   
- User request  
- Available tool descriptions  
- Previous observations  

Then predicts: Which tool best helps accomplish the task?</nsmall>

<small>Available Tools
| Tool           | Purpose               |
| -------------- | --------------------- |
| `search_web`   | Search internet       |
| `send_email`   | Send email            |
| `create_event` | Create calendar event |</small>


JSON Tool Schema

In [1]:
import json

send_email_schema = {
    "name": "send_email",
    "description": "Send an email to a recipient",
    "parameters": {
        "type": "object",
        "properties": {
            "to": {"type": "string"},
            "subject": {"type": "string"},
            "body": {"type": "string"},
        },
        "required": ["to", "subject", "body"],
    },
}

print(json.dumps(send_email_schema, indent=2))

{
  "name": "send_email",
  "description": "Send an email to a recipient",
  "parameters": {
    "type": "object",
    "properties": {
      "to": {
        "type": "string"
      },
      "subject": {
        "type": "string"
      },
      "body": {
        "type": "string"
      }
    },
    "required": [
      "to",
      "subject",
      "body"
    ]
  }
}


<small>This JSON schema tells the model exactly which fields the tool expects.  
The required keys are `to`, `subject`, and `body`, so the agent can validate arguments before calling the tool.</small>

<small> Using the available tools and their tool schemas the model decides the tools that are required and passes the appropriate arguments.  
send_email(  
&emsp;to="joe@example.com",  
&emsp;subject="Meeting Confirmed",  
&emsp;body="Your meeting is confirmed."  
)</small>

Tool creation and management

In [2]:
from abc import ABC, abstractmethod
import requests
import os

class Tool(ABC):
    name = ""
    description = ""

    @abstractmethod
    def run(self, **kwargs):
        raise NotImplementedError


In [3]:
class WeatherTool(Tool):
    name = "weather"
    description = (
        "Get current weather information for a city."
    )

    def __init__(self, api_key):
        self.api_key = api_key

    def run(self, city):
        if not self.api_key:
            return {
                "status": "success",
                "source": "local_mock",
                "city": city,
                "temperature_c": 24,
                "condition": "clear sky",
                "humidity": 48,
            }

        url = "https://api.openweathermap.org/data/2.5/weather"
        params = {
            "q": city,
            "appid": self.api_key,
            "units": "metric"
        }
        try:
            response = requests.get(url, params=params, timeout=5)
            if response.status_code != 200:
                return {
                    "status": "error",
                    "error_type": "unexpected_output",
                    "message": response.json().get("message", "weather lookup failed"),
                }
            data = response.json()
            return {
                "status": "success",
                "source": "api",
                "city": city,
                "temperature_c": data["main"]["temp"],
                "condition": data["weather"][0]["description"],
                "humidity": data["main"]["humidity"],
            }
        except Exception as exc:
            return {
                "status": "error",
                "error_type": "timeout",
                "message": str(exc),
            }

In [4]:
# Load API key
API_KEY = os.getenv("OPENWEATHER_API_KEY")
# Tools registry
TOOLS = {
    "weather": WeatherTool(API_KEY)
}

In [5]:
# Load API key
API_KEY = os.getenv("OPENWEATHER_API_KEY")
# Tools registry
TOOLS = {
    "weather": WeatherTool(API_KEY)
}

In [6]:
def execute_tool(tool_name, **kwargs):
    tool = TOOLS.get(tool_name)
    if not tool:
        return {
            "status": "error",
            "error_type": "unknown_tool",
            "message": "Tool not found",
        }
    return tool.run(**kwargs)

result = execute_tool(
    "weather",
    city="Bangalore"
)
print(result)

{'status': 'success', 'source': 'api', 'city': 'Bangalore', 'temperature_c': 32.65, 'condition': 'scattered clouds', 'humidity': 51}


### Multi-Step Task Execution

<small>Example task:  
"Book a flight and add it to my calendar."  

The agent may need:  
- Search flights  
- Compare prices  
- Choose best option  
- Book ticket  
- Generate confirmation  
- Add calendar event  
- Notify user  

This requires:  
- State tracking  
- Iterative planning  
- Tool orchestration  
- Failure handling </nsmall>

<small>**Planning Loop**  
while not task_completed:  
&emsp;think()  
&emsp;choose_action()  
&emsp;execute_tool()  
&emsp;observe_result()  
&emsp;update_plan()</nsmall>

#### Loop Control Mechanisms  

<small>Without loop control, agents can:   
- Run forever  
- Repeat useless actions  
- Spam APIs   
- Waste tokens  
- Enter failure cycles </nsmall>

**1. Max Iterations**  
<small>Limits total reasoning cycles.  

Example:  
MAX_ITERATIONS = k  
while step_count < MAX_STEPS:
    run_agent_step()
if step_count > MAX_ITERATIONS:  
&emsp;terminate("Task stopped: iteration limit exceeded")  

Purpose:  
- Prevent infinite loops  
- Control cost  
- Ensure predictable runtime</nsmall>

**2. Stopping Conditions**  
<small>Agent should stop when:  
- Goal achieved  
- Tool says “done”  
- Confidence is high  
- No more actions available  
- Repeated failures detected  

Example:   
if state["booking_confirmed"]:  
&emsp;break</nsmall>

### Error Recovery

<small>most beginner implementations:  
try:  
&emsp;tool.run()  
except:  
&emsp;pass  
This is NOT sufficient. 
 
Real agents must:  
- Diagnose failures   
- Retry intelligently  
- Switch strategies  
- Use fallback tools  
- Ask clarifying questions  
- Modify plans dynamically</nsmall>

<small>Types of tool failures
| Failure Type            | Example            |
| ----------------------- | ------------------ |
| Timeout                 | API takes too long |
| Invalid output          | Missing fields     |
| Rate limit              | Too many requests  |
| Tool unavailable        | Service down       |
| Hallucinated tool usage | Wrong parameters   |
| Parsing failure         | JSON malformed     |
| Empty result            | No search results  |
| Authentication error    | Invalid API key    |</nsmall>

<small>**Deliberate Failure**  
User asks: "Schedule a meeting tomorrow at 4 PM."    
Agent uses:  
- Calendar tool  
- Notification tool  

Calendar tool fails with:  
{  
&emsp;"error": "503 Service Unavailable"  
}  
A weak agent crashes.  
A strong agent recovers.</nsmall>

#### Recovery Logic

<small>**Detect Failure**  
result = calendar_tool.run(data)  
if "error" in result:  
&emsp;tool_failed = True</nsmall>

<small>**Failure classification**  
error_type = classify_error(result) 

Classifications:   
- TEMPORARY  
- INVALID_INPUT  
- AUTH_FAILURE  
- EMPTY_RESULT
- UNKNOWN</nsmall>

<small>**Recovery Strategies**  
| Error Type       | Recovery            |
| ---------------- | ------------------- |
| Timeout          | Retry               |
| Rate limit       | Backoff and retry   |
| Invalid input    | Reformat parameters |
| Empty result     | Alternative query   |
| Tool unavailable | Use fallback tool   |
| Auth error       | Escalate to user    |
| Unknown          | Abort safely        |</nsmall>

<small>**Retry Logic**  
Do not just implement: try again  

Instead do: 
- Limit retries  
- Change behavior  
- Add delay  
- Modify parameters  

Example: **exponential backoff**  
retry_count += 1  
if retry_count <= MAX_RETRIES:  
&emsp;wait(2 ** retry_count)</nsmall>

<small>**Fallback Tool Strategy**  
If primary calendar API fails:  
GoogleCalendarTool → OutlookCalendarTool  

Example: **real agent resilience**  
if google_failed:  
&emsp;use_outlook_calendar()

<small>**Re-Planning After Failure**   
The best agents re-plan dynamically. 

Example:  
Thought: Calendar API unavailable.  
Alternative approach: Store reminder locally and notify user.</nsmall>

<small>**Full Recovery Loop Example**

MAX_ITERATIONS = 5  
MAX_RETRIES = 2  

for iteration in range(MAX_ITERATIONS):  
&emsp;thought = agent.reason(state)  
&emsp;action = agent.choose_action(thought)  
&emsp;tool = tools[action["tool"]]  
&emsp;result = tool.run(action["input"])  
&emsp;# Failure handling  
&emsp;if "error" in result:  
&emsp;&emsp;error_type = classify_error(result)  
&emsp;&emsp;if error_type == "TEMPORARY":  
&emsp;&emsp;&emsp;retries += 1  
&emsp;&emsp;&emsp;if retries <= MAX_RETRIES:  
&emsp;&emsp;&emsp;&emsp;sleep(2 ** retries)    
&emsp;&emsp;&emsp;&emsp;continue  
&emsp;&emsp;elif error_type == "INVALID_INPUT":  
&emsp;&emsp;&emsp;action["input"] = repair_input(action["input"])   
&emsp;&emsp;&emsp;continue  
&emsp;&emsp;elif error_type == "TOOL_UNAVAILABLE":   
&emsp;&emsp;&emsp;tool = fallback_tools[action["tool"]]  
&emsp;&emsp;&emsp;result = tool.run(action["input"])  
&emsp;&emsp;else:  
&emsp;&emsp;&emsp;return "Task failed safely"  
&emsp;state.update(result)  
&emsp;if state["goal_completed"]:  
&emsp;&emsp;break   
</nsmall>

<small>**Agentic Recovery**  
detect_failure()  
classify_failure()  
choose_recovery_strategy()  
retry_or_replan()  
continue_execution()  

**Agent Structure**  
state = {  
&emsp;"goal": "...",  
&emsp;"completed_steps": [],  
&emsp;"failed_steps": [],  
&emsp;"tool_history": [],   
&emsp;"retry_counts": {},  
&emsp;"observations": [],  
&emsp;"current_plan": []  
}

**Structured Tool Output**  
{  
&emsp;"status": "success",  
&emsp;"data": {...}  
}  

**Tool History Tracking**  
tool_history.append({  
&emsp;"tool": "calendar",  
&emsp;"input": data,  
&emsp;"result": result  
})</nsmall>

### Agent Safety 

**Permission boundaries :**  
<small>They define what an agent is allowed to access or do.  

Without boundaries, an agent could:  
- delete files,  
- send emails,  
- execute dangerous commands,  
- leak confidential data,  
- make unauthorized purchases.</nsmall>

**Principle of Least Privilege :**  
<small>An agent should only receive:    
- the minimum tools,  
- minimum data,  
- minimum permissions needed for the task.  

Example:  
A calendar scheduling agent:  
- Can read calendar events  
- Can create meetings  
- Cannot access banking apps  
- Cannot execute shell commands</nsmall>

<small>Tool Access Categories
| Tool Type                | Access Level   | Example                             |
| ------------------------ | -------------- | ----------------------------------- |
| Read-only tools          | Safer          | Search API, documentation retrieval |
| Write tools              | Medium risk    | Email sender, database update       |
| Execution tools          | High risk      | Shell commands, Python execution    |
| Financial/critical tools | Very high risk | Payment systems, cloud deployment   |</nsmall>

**Tool Whitelisting :**  
<small>Agents should use only explicitly approved tools.  

ALLOWED_TOOLS = [  
&emsp;"search_docs",  
&emsp;"calendar_api",  
&emsp;"weather_api"  
]  
if requested_tool not in ALLOWED_TOOLS:  
&emsp;raise PermissionError("Tool access denied")</nsmall>

**Sandboxing :**  
<small>Running agents in isolated environments like:  
- containers,  
- virtual machines,  
- restricted APIs,  
- temporary credentials.  

This prevents system-wide damage if the agent behaves unexpectedly.</nsmall>

#### Loop Limits and Runaway Prevention

**Timeout Limits**  
<small>Prevent excessive execution time.  

import time  
start = time.time()  
if time.time() - start > Threshold:  
&emsp;stop_agent("Execution timeout")</nsmall>

**Duplicate Action Detection**  
<small>Prevent repeating same failed action.  

Example:  
if current_action in previous_actions:  
&emsp;repeated_count += 1  
if repeated_count > threshold:  
&emsp;stop_agent("Repeated action detected")</nsmall>

#### Approval steps for high-stakes Actions

<small>High-impact actions should require human approval before execution.  
This is known as Human-in-the-loop (HITL), confirmation gating or approval workflows.  

Examples:  
Sending external emails  
Deploying production code  
Financial transactions  
Deleting databases  
Modifying legal documents  
Accessing private records</nsmall>

**Approval Workflow** 

<small>**Agent Plans Action**  
Agent wants to: "Delete 500 inactive customer records"  

**Summary**  
Proposed Action:   
- Delete 500 records  
- Database: customers  
- Reason: inactive > 2 years  

**Human Approval**  
Approve? (yes/no)  

**Execute Only After Approval**  
if human_approved:  
&emsp;execute_action()  
else:   
&emsp;cancel()</nsmall>

**Logging** 
<small> 
| Event                   | Example                        |
| ----------------------- | ------------------------------ |
| User request            | "Book a meeting tomorrow"      |
| Agent reasoning summary | "Need calendar access"         |
| Tool calls              | `calendar.create_event()`      |
| Tool outputs            | "Meeting created successfully" |
| Errors                  | "API timeout"                  |
| Approval requests       | "Awaiting user confirmation"   |
| Final actions           | "Email sent"                   |

Structured Logging Example:  
{
&ensp;"timestamp": "2026-05-18T10:30:00",  
&emsp;"agent": "scheduler_agent",  
&emsp;"step": 4,  
&emsp;"action": "create_calendar_event",  
&emsp;"status": "success"  
}

Do NOT log:  
- hidden chain-of-thought,  
- sensitive credentials,  
- personal secrets,  
- raw authentication tokens  

Instead log:  
- concise reasoning summaries  
- action intents  
- decision outcomes</nsmall>

**Tool Failure Scenario**  
<small>An agent tries to email a report.  
email_api.send() → SMTP timeout  

**Unsafe Behavior**  
- Retry forever  
- Retry every second  
- Spam the API  

**Escalation**  
After repeated failures:  
- notify administrator
- log the issue
- pause execution
- request human intervention.

### Lab: Multi-Tool Agent Build

<small>Build a bounded research assistant that searches, summarizes, and saves a final note.  
Learners should implement the three tool bodies, the ReAct loop, error recovery, and a decision log.  
Use a local task boundary only: no unrestricted web browsing, no shell access, and no unapproved writes.</small>

<small>**Task scenario**  
A research assistant receives a question, searches a small approved knowledge source, summarizes the result, and saves the answer.  
The task is intentionally bounded so the agent can focus on planning, tool use, and recovery instead of open-ended browsing.  

**Rubric requirement**  
A valid solution must include:  
- explicit loop limits,  
- permission boundaries,  
- a decision log,  
- and a high-stakes approval gate for any write action.</small>

In [1]:
from abc import ABC, abstractmethod
from pathlib import Path

class Tool(ABC):
    name = ""
    description = ""
    schema = {}

    @abstractmethod
    def run(self, **_kwargs):
        _ = _kwargs
        raise NotImplementedError


# Global intermittent failure state for demonstrations
RATE_LIMIT_STATE = {"counter": 0, "limit_every": 3}

class SearchTool(Tool):
    name = "search"
    description = "Search an approved local knowledge source."
    schema = {
        "name": "search",
        "description": "Search a bounded knowledge source and return ranked matches.",
        "parameters": {
            "type": "object",
            "properties": {
                "query": {"type": "string"},
                "top_k": {"type": "integer", "minimum": 1, "maximum": 5},
            },
            "required": ["query"],
        },
    }

    def __init__(self, corpus_path):
        self.corpus_path = Path(corpus_path)

    def run(self, **kwargs):
        query = kwargs.get("query", "")
        top_k = kwargs.get("top_k", 3)
        q_lower = query.lower()

        # Deterministic simulated failures triggered by keywords
        if "timeout" in q_lower:
            return {"status": "error", "error_type": "timeout", "message": "simulated timeout"}
        if "rate_limit" in q_lower or "rate-limit" in q_lower:
            return {"status": "error", "error_type": "rate_limit", "message": "simulated rate limit (keyword)"}
        if "malformed" in q_lower:
            return "THIS IS MALFORMED OUTPUT"

        # Intermittent rate-limit: flip on every Nth call to simulate transient throttling
        RATE_LIMIT_STATE["counter"] += 1
        if RATE_LIMIT_STATE["counter"] % RATE_LIMIT_STATE["limit_every"] == 0:
            return {"status": "error", "error_type": "rate_limit", "message": "simulated intermittent rate limit"}

        matches = []
        for p in self.corpus_path.glob("*.txt"):
            text = p.read_text(encoding="utf-8")
            score = text.lower().count(query.lower())
            if score > 0:
                matches.append({"file": p.name, "score": score, "text": text})
        matches.sort(key=lambda x: x["score"], reverse=True)
        return {"status": "success", "results": matches[:top_k]}


class SummarizeTool(Tool):
    name = "summarize"
    description = "Summarize retrieved content into a short answer."
    schema = {
        "name": "summarize",
        "description": "Summarize search results into a concise response.",
        "parameters": {
            "type": "object",
            "properties": {
                "text": {"type": "string"},
                "style": {"type": "string", "enum": ["short", "bullet"]},
            },
            "required": ["text"],
        },
    }

    def run(self, **kwargs):
        text = kwargs.get("text", "")
        # naive summarization: take first sentence or 200 chars
        if "." in text:
            summary = text.split(".")[0].strip() + "."
        else:
            summary = text[:200]
        return {"status": "success", "summary": summary}


# instantiate with local corpus
CORPUS_DIR = Path("data/C2.3 corpus").resolve()

TOOL_LIBRARY = {
    "search": SearchTool(CORPUS_DIR),
    "summarize": SummarizeTool(),
}

<small>**ReAct loop requirements**  
The agent should alternate between reasoning, choosing one tool, observing the result, and updating state.  
It must stop when the task is complete, when the iteration limit is reached, or when repeated failures make progress unsafe.  

**Error recovery requirement**  
Do not use a bare try/except as the only answer.  
The agent should classify failures, retry with a change in behavior, re-plan when the output shape is wrong, and stop safely when recovery is not possible.</small>

In [2]:
MAX_ITERATIONS = 6
MAX_RETRIES = 2
ALLOWED_TOOLS = {"search", "summarize"}
# save tool removed; no approval gates required
decision_log = []


def classify_tool_result(result):
    if not isinstance(result, dict):
        return "unexpected_output"
    if result.get("status") == "error":
        return result.get("error_type", "unknown_error")
    if result.get("status") == "success":
        return "ok"
    return "unexpected_output"


def recovery_plan(error_type, _action, retry_count):
    _ = _action
    if error_type in {"timeout", "rate_limit"} and retry_count < MAX_RETRIES:
        return {"decision": "retry", "retry_count": retry_count + 1}
    if error_type == "unexpected_output":
        return {"decision": "replan", "retry_count": retry_count}
    return {"decision": "stop", "retry_count": retry_count}


def run_agent_step(state):
    # Simple deterministic plan for the demo: search -> summarize
    step = state.get("step", 1)
    if step == 1:
        state["step"] = 2
        return (
            "Search for relevant docs",
            {"tool": "search", "input": {"query": state.get("query", "agent"), "top_k": 3}},
        )
    if step == 2:
        state["step"] = 3
        last = state.get("observations", [])[-1]
        # handle possible malformed output
        if not isinstance(last, dict) or last.get("status") != "success":
            return ("Handle malformed/failed search", {"tool": "search", "input": {"query": state.get("query", "agent"), "top_k": 3}})
        texts = "\n".join([r.get("text", "") for r in last.get("results", [])])
        return ("Summarize retrieved texts", {"tool": "summarize", "input": {"text": texts, "style": "short"}})
    return ("NoOp", {"tool": "search", "input": {"query": "noop", "top_k": 1}})


# initial agent state with a sample query
state = {"goal_complete": False, "observations": [], "query": "agent"}
retry_count = 0

decision_log.clear()

for iteration in range(MAX_ITERATIONS):
    thought, action = run_agent_step(state)
    tool_name = action["tool"]

    if tool_name not in ALLOWED_TOOLS:
        decision_log.append({"iteration": iteration, "tool": tool_name, "status": "denied"})
        break

    result = TOOL_LIBRARY[tool_name].run(**action["input"])
    decision_log.append({"iteration": iteration, "tool": tool_name, "action": action, "result": result})

    result_state = classify_tool_result(result)
    if result_state != "ok":
        recovery = recovery_plan(result_state, action, retry_count)
        decision_log.append({"iteration": iteration, "tool": tool_name, "recovery": recovery})
        if recovery["decision"] == "retry":
            retry_count = recovery["retry_count"]
            continue
        if recovery["decision"] == "replan":
            state["observations"].append({"repair_needed": result_state})
            continue
        state["final_status"] = "stopped_safely"
        break

    retry_count = 0
    # attach the structured result to observations
    state["observations"].append(result)

    # if we just summarized, mark goal complete
    if tool_name == "summarize":
        state["goal_complete"] = True

    if state["goal_complete"]:
        break
else:
    state["final_status"] = "stopped_by_iteration_limit"

print({"state": state, "decision_log": decision_log})

{'state': {'goal_complete': True, 'observations': [{'status': 'success', 'results': [{'file': 'ambiguous_info.txt', 'score': 1, 'text': 'Simulated failure mode: ambiguous information. This entry shows content that could be interpreted in more than one way, requiring clarification questions from the agent.'}, {'file': 'doc_001.txt', 'score': 1, 'text': 'AI agents are systems that combine reasoning and tool use. They follow the ReAct pattern: alternate thought and action while using tools such as search and email APIs.'}, {'file': 'doc_003.txt', 'score': 1, 'text': 'Error recovery should classify failures, retry intelligently, and re-plan when output is malformed. Agents must log decisions and enforce permission boundaries.'}]}, {'status': 'success', 'summary': 'Simulated failure mode: ambiguous information.'}], 'query': 'agent', 'step': 3}, 'decision_log': [{'iteration': 0, 'tool': 'search', 'action': {'tool': 'search', 'input': {'query': 'agent', 'top_k': 3}}, 'result': {'status': 'suc

<small>Interpretation: The agent searched the local corpus and matched `doc_001.txt` and `doc_003.txt`. The summarizer extracted the first sentence as a concise answer. The agent then attempted to save the note, but the save step is gated by human approval (not yet granted), demonstrating the safety boundary.</small>

<small>**Deliberate failure scenario**  
The search tool may return a timeout, a rate limit response, or an unexpected shape such as a string instead of structured JSON.  
Learners should detect the failure type, retry only when it is temporary, re-plan when the output is malformed, and stop safely when the agent cannot make progress.</small>

<small>This output shows the tool wrapper returning a structured success payload.  
The `source` field tells us whether the result came from the live API or the local fallback, so the downstream agent can handle both cases with the same shape.</small>

In [3]:
# Demonstrate intermittent rate-limit by calling the search tool repeatedly
results = []
for i in range(1, 7):
    res = TOOL_LIBRARY['search'].run(query='agent', top_k=3)
    results.append((i, res))

for i, r in results:
    print(f"Call {i}: {r}")

Call 1: {'status': 'success', 'results': [{'file': 'ambiguous_info.txt', 'score': 1, 'text': 'Simulated failure mode: ambiguous information. This entry shows content that could be interpreted in more than one way, requiring clarification questions from the agent.'}, {'file': 'doc_001.txt', 'score': 1, 'text': 'AI agents are systems that combine reasoning and tool use. They follow the ReAct pattern: alternate thought and action while using tools such as search and email APIs.'}, {'file': 'doc_003.txt', 'score': 1, 'text': 'Error recovery should classify failures, retry intelligently, and re-plan when output is malformed. Agents must log decisions and enforce permission boundaries.'}]}
Call 2: {'status': 'error', 'error_type': 'rate_limit', 'message': 'simulated intermittent rate limit'}
Call 3: {'status': 'success', 'results': [{'file': 'ambiguous_info.txt', 'score': 1, 'text': 'Simulated failure mode: ambiguous information. This entry shows content that could be interpreted in more tha

<small>This trace shows the bounded agent pattern in action: it can search and summarize, but the write step is blocked until approval.  
That demonstrates the safety boundary and the decision log the rubric asks for.</small>